# STEP 1 — SCRAPING ULASAN ADAKAMI
## Tesis: Analisis Sentimen Ulasan Pengguna Aplikasi AdaKami di Google Play Store

**Tujuan cell ini:** Mengambil ulasan pengguna AdaKami dari Google Play Store periode Januari–Desember 2025, lalu menyimpannya ke file CSV.

**Output yang diharapkan:** File `ulasan_adakami_2025_raw.csv` berisi minimal 5.000 baris ulasan.

---
> ⚠️ **Jalankan cell secara berurutan dari atas ke bawah (Shift+Enter)**

### 📦 Cell 1 — Install Library
Jalankan sekali saja. Kalau sudah pernah install, skip cell ini.

In [ ]:
!pip install google-play-scraper -q
print("✅ Library berhasil diinstall")

### 📚 Cell 2 — Import Library

In [ ]:
from google_play_scraper import reviews, Sort
import pandas as pd
from datetime import datetime
import time

print("✅ Import berhasil")
print(f"📅 Scraping dimulai: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

### ⚙️ Cell 3 — Konfigurasi Scraping

Tidak perlu mengubah apapun di cell ini. Jalankan saja.

In [ ]:
# ============================================================
# KONFIGURASI — tidak perlu diubah
# ============================================================
APP_ID       = 'com.adakami.dana.kredit.pinjaman'  # ID aplikasi AdaKami di Play Store
LANG         = 'id'                                 # Bahasa Indonesia
COUNTRY      = 'id'                                 # Negara Indonesia
BATCH_SIZE   = 200                                  # Ulasan per request (max 200)
TARGET_TOTAL = 15000                                # Ambil lebih banyak, nanti difilter per tahun

# Filter tanggal: hanya ambil ulasan tahun 2025
TANGGAL_MULAI = datetime(2025, 1, 1)
TANGGAL_AKHIR = datetime(2025, 12, 31, 23, 59, 59)

NAMA_FILE = 'ulasan_adakami_2025_raw.csv'

print(f"🎯 Target aplikasi : {APP_ID}")
print(f"📅 Periode data    : {TANGGAL_MULAI.date()} s/d {TANGGAL_AKHIR.date()}")
print(f"📦 Batch size      : {BATCH_SIZE} ulasan/request")

### 🚀 Cell 4 — Jalankan Scraping

> ⏱️ **Estimasi waktu: 5–15 menit** tergantung jumlah ulasan dan koneksi internet.
> Kamu akan melihat progress bar selama proses berjalan. **Jangan tutup tab Colab.**

In [ ]:
semua_ulasan = []
continuation_token = None
sudah_lewat_2025 = False

print("🔄 Memulai scraping...\n")

batch_ke = 0
while len(semua_ulasan) < TARGET_TOTAL and not sudah_lewat_2025:
    batch_ke += 1

    try:
        hasil, continuation_token = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,       # Urutkan dari terbaru
            count=BATCH_SIZE,
            continuation_token=continuation_token
        )
    except Exception as e:
        print(f"⚠️  Error pada batch {batch_ke}: {e}")
        print("⏳ Menunggu 10 detik lalu mencoba lagi...")
        time.sleep(10)
        continue

    if not hasil:
        print("⚠️  Tidak ada ulasan lagi dari server. Scraping selesai.")
        break

    for ulasan in hasil:
        tgl = ulasan['at']
        # Pastikan datetime aware vs naive konsisten
        if tgl.tzinfo is not None:
            tgl = tgl.replace(tzinfo=None)

        if tgl < TANGGAL_MULAI:
            # Ulasan sudah lebih lama dari Januari 2025 — berhenti
            sudah_lewat_2025 = True
            break

        if TANGGAL_MULAI <= tgl <= TANGGAL_AKHIR:
            semua_ulasan.append({
                'review_id'  : ulasan['reviewId'],
                'username'   : ulasan['userName'],
                'rating'     : ulasan['score'],
                'teks_ulasan': ulasan['content'],
                'tanggal'    : tgl.strftime('%Y-%m-%d'),
                'bulan'      : tgl.month,
                'tahun'      : tgl.year,
                'thumbs_up'  : ulasan['thumbsUpCount'],
                'versi_app'  : ulasan.get('reviewCreatedVersion', None)
            })

    print(f"  Batch {batch_ke:>3} | Terkumpul: {len(semua_ulasan):>5} ulasan | "
          f"Ulasan terakhir: {hasil[-1]['at'].strftime('%Y-%m-%d') if hasil else '-'}")

    if continuation_token is None:
        print("✅ Semua ulasan sudah diambil (tidak ada token lanjutan).")
        break

    time.sleep(1)  # Jeda 1 detik antar request agar tidak diblokir

print(f"\n{'='*50}")
print(f"✅ Scraping selesai!")
print(f"📊 Total ulasan terkumpul (2025): {len(semua_ulasan):,}")

### 💾 Cell 5 — Simpan ke CSV & Preview Data

In [ ]:
# Buat DataFrame dari hasil scraping
df_raw = pd.DataFrame(semua_ulasan)

# Hapus duplikat berdasarkan review_id
df_raw = df_raw.drop_duplicates(subset='review_id').reset_index(drop=True)

# Simpan ke CSV
NAMA_FILE = 'ulasan_adakami_2025_raw.csv'
df_raw.to_csv(NAMA_FILE, index=False, encoding='utf-8-sig')

print(f"✅ Data tersimpan di: {NAMA_FILE}")
print(f"\n{'='*50}")
print(f"📊 RINGKASAN DATA")
print(f"{'='*50}")
print(f"Total ulasan       : {len(df_raw):,}")
print(f"Kolom tersedia     : {list(df_raw.columns)}")
print(f"Rentang tanggal    : {df_raw['tanggal'].min()} s/d {df_raw['tanggal'].max()}")
print(f"\n📋 Distribusi Rating:")
print(df_raw['rating'].value_counts().sort_index().to_string())
print(f"\n📅 Distribusi per Bulan:")
print(df_raw['bulan'].value_counts().sort_index().to_string())

### 🔍 Cell 6 — Cek Sampel Ulasan

In [ ]:
# Tampilkan 5 contoh ulasan acak
print("📝 CONTOH ULASAN (5 acak):\n")
sample = df_raw.sample(5, random_state=42)[['tanggal', 'rating', 'teks_ulasan']]
for i, row in sample.iterrows():
    print(f"[{row['tanggal']}] ⭐{row['rating']} — {row['teks_ulasan'][:150]}...")
    print()

# Cek ulasan kosong
kosong = df_raw['teks_ulasan'].isna().sum() + (df_raw['teks_ulasan'] == '').sum()
print(f"⚠️  Ulasan dengan teks kosong : {kosong}")
print(f"✅ Ulasan dengan teks terisi  : {len(df_raw) - kosong}")
print(f"\n📁 File tersimpan di: {NAMA_FILE}")

In [ ]:
# Tampilkan 5 contoh ulasan acak
print("📝 CONTOH ULASAN (5 acak):\n")
sample = df_raw.sample(5, random_state=42)[['tanggal', 'rating', 'teks_ulasan']]
for i, row in sample.iterrows():
    print(f"[{row['tanggal']}] ⭐{row['rating']} — {row['teks_ulasan'][:150]}...")
    print()

# Cek apakah ada ulasan kosong
kosong = df_raw['teks_ulasan'].isna().sum() + (df_raw['teks_ulasan'] == '').sum()
print(f"\n⚠️  Ulasan dengan teks kosong: {kosong}")
print(f"✅ Ulasan dengan teks terisi : {len(df_raw) - kosong}")

In [ ]:
from google.colab import files
files.download(NAMA_FILE)
print(f"✅ File {NAMA_FILE} sedang didownload ke komputer kamu")

---
## ✅ STEP 1 SELESAI — Checklist Sebelum Lanjut

Sebelum pindah ke Step 2 (EDA), pastikan:

- [ ] Cell 4 selesai tanpa error merah
- [ ] Total ulasan terkumpul **≥ 5.000**
- [ ] Rentang tanggal benar: **2025-01-xx s/d 2025-12-xx**
- [ ] File `ulasan_adakami_2025_raw.csv` sudah ada di Colab (panel kiri Files)
- [ ] Sudah download CSV ke komputer (backup)

**Jika total ulasan < 5.000:** Screenshot output Cell 4 dan tunjukkan ke saya — kita troubleshoot bersama.

**Jika sudah ≥ 5.000:** Lanjut ke `step2_eda.ipynb` 🎉